In [2]:
import openeo

connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


In [ ]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [112.7511784, -7.027599],
            [112.753737, -7.028691],
            [112.7526367, -7.0323882],
            [112.7475858, -7.0294805],
            [112.7511784, -7.027599]
        ]
    ]
}

In [ ]:
# Konfigurasi parameter pengambilan data
pollutants = ['NO2', 'CO', 'SO2', 'CH4']
start_date = "2025-08-30"
end_date = "2026-08-30"

# Area of Interest (Bangkalan, Pulau Madura)
spatial_extent = {
    "west": 112.7475858,
    "south": -7.0323882,
    "east": 112.753737,
    "north": -7.027599
}

print("=" * 70)
print("KONFIGURASI PENGAMBILAN DATA SENTINEL-5P")
print("=" * 70)
print(f"Periode: {start_date} hingga {end_date}")
print(f"Lokasi: Bangkalan, Pulau Madura")
print(f"Bounding Box:")
print(f"  - Latitude:  {spatial_extent['south']:.4f} - {spatial_extent['north']:.4f}")
print(f"  - Longitude: {spatial_extent['west']:.4f} - {spatial_extent['east']:.4f}")
print(f"Pollutant yang akan diambil: {', '.join(pollutants)}")
print("=" * 70)

KONFIGURASI PENGAMBILAN DATA SENTINEL-5P
Periode: 2025-08-30 hingga 2026-08-30
Lokasi: Telukdalam, Pulau Nias
Bounding Box:
  - Latitude:  0.5569 - 0.5790
  - Longitude: 97.8086 - 97.8312
Pollutant yang akan diambil: NO2, CO, SO2, NH3


In [ ]:
# Fungsi untuk mengambil dan memproses data setiap pollutant
def process_pollutant(connection, pollutant, spatial_extent, aoi, start_date, end_date):
    """
    Mengambil data Sentinel-5P untuk satu pollutant,
    agregasi temporal (daily) dan spatial (mean),
    kemudian execute batch job.
    """
    print(f"\n{'='*70}")
    print(f"MEMPROSES: {pollutant}")
    print(f"{'='*70}")

    try:
        print(f"1. Loading collection SENTINEL_5P_L2 dengan band {pollutant}...")
        s5p_data = connection.load_collection(
            "SENTINEL_5P_L2",
            temporal_extent=[start_date, end_date],
            spatial_extent=spatial_extent,
            bands=[pollutant],
        )
        print("   ✓ Collection berhasil dimuat")

        print("2. Aggregasi temporal (daily mean)...")
        s5p_data = s5p_data.aggregate_temporal_period(reducer="mean", period="day")
        print("   ✓ Aggregasi temporal selesai")

        print("3. Aggregasi spatial (mean across AOI)...")
        s5p_data = s5p_data.aggregate_spatial(reducer="mean", geometries=aoi)
        print("   ✓ Aggregasi spatial selesai")

        print("4. Execute batch job...")
        job_title = f"{pollutant} Bangkalan {start_date} to {end_date}"
        output_file = f"{pollutant}_bangkalan_2025-2026.nc"

        job = s5p_data.execute_batch(title=job_title, outputfile=output_file)
        print(f"   ✓ Batch job '{job_title}' berhasil diexecute")
        print(f"   ✓ Output file: {output_file}")

        return job, output_file

    except Exception as e:
        print(f"   ✗ Error: {str(e)}")
        return None, None

In [ ]:
# Jalankan process untuk semua pollutant
results = {}

for pollutant in pollutants:
    job, output_file = process_pollutant(
        connection, 
        pollutant, 
        spatial_extent, 
        aoi, 
        start_date, 
        end_date
    )
    
    if job is not None:
        results[pollutant] = {
            'job': job,
            'output_file': output_file,
            'status': 'submitted'
        }
    else:
        results[pollutant] = {
            'job': None,
            'output_file': None,
            'status': 'error'
        }

# Ringkasan hasil
print(f"\n{'='*70}")
print("RINGKASAN SUBMISSION BATCH JOBS")
print(f"{'='*70}")
for pollutant, info in results.items():
    status_symbol = "✓" if info['status'] == 'submitted' else "✗"
    print(f"{status_symbol} {pollutant:6s}: {info['status']:12s} -> {info['output_file']}")
print(f"{'='*70}")


MEMPROSES: NO2
1. Loading collection SENTINEL_5P_L2 dengan band NO2...
   ✓ Collection berhasil dimuat
2. Aggregasi temporal (daily mean)...
   ✓ Agregasi temporal selesai
3. Agregasi spatial (mean across AOI)...
   ✓ Agregasi spatial selesai
4. Execute batch job...
0:00:00 Job 'j-26083021064540f0816ba8865190c62c': send 'start'
0:00:02 Job 'j-26083021064540f0816ba8865190c62c': queued (progress 0%)
0:00:08 Job 'j-26083021064540f0816ba8865190c62c': queued (progress 0%)
0:00:15 Job 'j-26083021064540f0816ba8865190c62c': queued (progress 0%)
0:00:23 Job 'j-26083021064540f0816ba8865190c62c': queued (progress 0%)
0:00:33 Job 'j-26083021064540f0816ba8865190c62c': running (progress N/A)
0:00:46 Job 'j-26083021064540f0816ba8865190c62c': running (progress N/A)
0:01:02 Job 'j-26083021064540f0816ba8865190c62c': running (progress N/A)
0:01:21 Job 'j-26083021064540f0816ba8865190c62c': running (progress N/A)
0:01:46 Job 'j-26083021064540f0816ba8865190c62c': running (progress N/A)
0:02:16 Job 'j-26083